In [11]:
# ============================================================
# WEEK 2 INTERNSHIP PROJECT
# TELCO CUSTOMER CHURN - PREDICTIVE MODELING
# ============================================================

import os
import glob
import warnings

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. PROJECT LOCATION
# ============================================================

PROJECT_FOLDER = os.getcwd()

VISUALIZATION_FOLDER = os.path.join(
    PROJECT_FOLDER,
    "visualization"
)

os.makedirs(
    VISUALIZATION_FOLDER,
    exist_ok=True
)


print("=" * 70)
print("TELCO CUSTOMER CHURN")
print("WEEK 2 - PREDICTIVE MODELING")
print("=" * 70)


# ============================================================
# 2. FIND ALL DATA FILES
# ============================================================

all_files = []

all_files += glob.glob(
    os.path.join(
        PROJECT_FOLDER,
        "**",
        "*.csv"
    ),
    recursive=True
)

all_files += glob.glob(
    os.path.join(
        PROJECT_FOLDER,
        "**",
        "*.xlsx"
    ),
    recursive=True
)

all_files += glob.glob(
    os.path.join(
        PROJECT_FOLDER,
        "**",
        "*.xls"
    ),
    recursive=True
)


# Remove files from unnecessary folders

all_files = [

    file for file in all_files

    if "visualization" not in file.lower()

    and ".git" not in file.lower()

]


if len(all_files) == 0:

    raise FileNotFoundError(
        "No CSV or Excel dataset was found inside the project."
    )


print("\nSearching for the Telco Customer Churn dataset...")


# ============================================================
# 3. FIND THE CORRECT TELCO DATASET
# ============================================================

selected_file = None
selected_data = None
selected_score = -1


for file in all_files:

    try:

        if file.lower().endswith(".csv"):

            temp_df = pd.read_csv(
                file,
                nrows=5
            )

        else:

            temp_df = pd.read_excel(
                file,
                nrows=5
            )

        # Clean column names temporarily

        temp_columns = [

            str(column)
            .strip()
            .lower()
            .replace(" ", "")
            .replace("_", "")
            for column in temp_df.columns

        ]

        score = 0


        # Telco-specific columns

        telco_columns = [

            "customerid",

            "gender",

            "seniorcitizen",

            "partner",

            "dependents",

            "tenure",

            "phoneservice",

            "multiplelines",

            "internetservice",

            "onlinesecurity",

            "onlinebackup",

            "deviceprotection",

            "techsupport",

            "streamingtv",

            "streamingmovies",

            "contract",

            "paperlessbilling",

            "paymentmethod",

            "monthlycharges",

            "totalcharges"

        ]


        for column in telco_columns:

            if column in temp_columns:

                score += 1


        # Churn-related columns

        churn_columns = [

            "churn",

            "churnlabel",

            "churnvalue",

            "customerstatus",

            "churnstatus",

            "churned",

            "attrition",

            "attritionflag"

        ]


        for column in churn_columns:

            if column in temp_columns:

                score += 20


        if score > selected_score:

            selected_score = score

            selected_file = file


    except Exception:

        continue


if selected_file is None:

    raise FileNotFoundError(
        "Could not find a readable dataset."
    )


print(
    "\nDataset selected:"
)

print(
    selected_file
)


# ============================================================
# 4. LOAD THE SELECTED DATASET
# ============================================================

if selected_file.lower().endswith(".csv"):

    df = pd.read_csv(
        selected_file
    )

else:

    df = pd.read_excel(
        selected_file
    )


print(
    "\nDataset loaded successfully."
)


print(
    "Rows:",
    df.shape[0]
)


print(
    "Columns:",
    df.shape[1]
)


# ============================================================
# 5. CLEAN COLUMN NAMES
# ============================================================

df.columns = (

    df.columns
    .astype(str)
    .str.strip()

)


print(
    "\nDataset columns:"
)


print(
    df.columns.tolist()
)


# ============================================================
# 6. FIND CHURN TARGET
# ============================================================

def normalise_name(name):

    return (

        str(name)
        .strip()
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")

    )


normalised_columns = {

    normalise_name(column): column

    for column in df.columns

}


churn_column = None


# Direct Churn column

possible_churn_names = [

    "churn",

    "churnlabel",

    "churnvalue",

    "churnstatus",

    "churned",

    "attrition",

    "attritionflag"

]


for name in possible_churn_names:

    if name in normalised_columns:

        churn_column = normalised_columns[name]

        break


# ============================================================
# 7. HANDLE CUSTOMER STATUS DATASET
# ============================================================

if churn_column is None:

    if "customerstatus" in normalised_columns:

        status_column = normalised_columns[
            "customerstatus"
        ]

        print(
            "\nCustomer Status column detected."
        )

        print(
            "Creating Churn target from Customer Status..."
        )

        df["Churn_Target"] = (

            df[status_column]
            .astype(str)
            .str.strip()
            .str.lower()
            .apply(

                lambda x:
                1
                if x == "churned"
                else 0

            )

        )

    else:

        # Search for any column containing churn

        possible_columns = [

            column

            for column in df.columns

            if "churn" in str(column).lower()

            or "attrition" in str(column).lower()

        ]


        if possible_columns:

            churn_column = possible_columns[0]


if churn_column is not None:

    print(
        "\nChurn column detected:"
    )

    print(
        churn_column
    )

    print(
        "\nOriginal churn values:"
    )

    print(
        df[churn_column].value_counts(
            dropna=False
        )
    )


    def convert_target(value):

        if pd.isna(value):

            return None


        text = (

            str(value)
            .strip()
            .lower()

        )


        if text in [

            "yes",
            "y",
            "1",
            "true",
            "churn",
            "churned"

        ]:

            return 1


        if text in [

            "no",
            "n",
            "0",
            "false",
            "not churn",
            "stayed",
            "active"

        ]:

            return 0


        try:

            number = float(value)

            if number == 1:

                return 1

            if number == 0:

                return 0

        except:

            pass


        return None


    df["Churn_Target"] = (

        df[churn_column]
        .apply(convert_target)

    )


# ============================================================
# 8. FINAL TARGET CHECK
# ============================================================

if "Churn_Target" not in df.columns:

    print(
        "\nThe selected dataset does not contain a usable churn target."
    )

    print(
        "\nAvailable columns:"
    )

    for column in df.columns:

        print(
            "-",
            column
        )

    raise ValueError(
        "The dataset selected does not contain a churn target."
    )


# Remove rows where target is unknown

df = df.dropna(
    subset=["Churn_Target"]
).copy()


df["Churn_Target"] = (

    df["Churn_Target"]
    .astype(int)

)


print(
    "\nFinal churn distribution:"
)


print(
    df["Churn_Target"].value_counts()
)


# ============================================================
# 9. REMOVE DUPLICATES
# ============================================================

print(
    "\nDuplicate rows before removal:"
)


print(
    df.duplicated().sum()
)


df = df.drop_duplicates().copy()


print(
    "Duplicate rows after removal:"
)


print(
    df.duplicated().sum()
)


# ============================================================
# 10. HANDLE TOTAL CHARGES
# ============================================================

total_charges_column = None


for column in df.columns:

    if normalise_name(column) == "totalcharges":

        total_charges_column = column

        break


if total_charges_column is not None:

    df[total_charges_column] = pd.to_numeric(

        df[total_charges_column],

        errors="coerce"

    )


    df[total_charges_column] = (

        df[total_charges_column]
        .fillna(
            df[total_charges_column].median()
        )

    )


# ============================================================
# 11. CREATE X AND Y
# ============================================================

y = df[
    "Churn_Target"
]


columns_to_remove = [

    "Churn_Target"

]


if churn_column is not None:

    columns_to_remove.append(
        churn_column
    )


X = df.drop(
    columns=columns_to_remove,
    errors="ignore"
).copy()


# Remove ID columns

id_columns = []


for column in X.columns:

    name = normalise_name(
        column
    )

    if name in [

        "customerid",

        "id",

        "customeridentifier"

    ]:

        id_columns.append(
            column
        )


if id_columns:

    X = X.drop(
        columns=id_columns
    )


# ============================================================
# 12. REMOVE COMPLETELY EMPTY COLUMNS
# ============================================================

X = X.dropna(
    axis=1,
    how="all"
)


# ============================================================
# 13. IDENTIFY FEATURE TYPES
# ============================================================

numerical_features = (

    X.select_dtypes(
        include="number"
    )
    .columns
    .tolist()

)


categorical_features = (

    X.select_dtypes(
        include=[
            "object",
            "category",
            "bool"
        ]
    )
    .columns
    .tolist()

)


print(
    "\nNumerical features:"
)

print(
    numerical_features
)


print(
    "\nCategorical features:"
)

print(
    categorical_features
)


# ============================================================
# 14. TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = (

    train_test_split(

        X,

        y,

        test_size=0.20,

        random_state=42,

        stratify=y

    )

)


print(
    "\nTraining records:",
    len(X_train)
)


print(
    "Testing records:",
    len(X_test)
)


# ============================================================
# 15. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(

    transformers=[

        (

            "categorical",

            OneHotEncoder(

                handle_unknown="ignore",

                sparse_output=False

            ),

            categorical_features

        ),

        (

            "numerical",

            "passthrough",

            numerical_features

        )

    ]

)


# ============================================================
# 16. DECISION TREE
# ============================================================

print(
    "\nTraining Decision Tree..."
)


decision_tree = Pipeline(

    steps=[

        (

            "preprocessor",

            preprocessor

        ),

        (

            "model",

            DecisionTreeClassifier(

                max_depth=5,

                random_state=42

            )

        )

    ]

)


decision_tree.fit(
    X_train,
    y_train
)


dt_predictions = decision_tree.predict(
    X_test
)


dt_probabilities = (

    decision_tree
    .predict_proba(X_test)[:, 1]

)


print(
    "Decision Tree completed."
)


# ============================================================
# 17. RANDOM FOREST
# ============================================================

print(
    "\nTraining Random Forest..."
)


random_forest = Pipeline(

    steps=[

        (

            "preprocessor",

            preprocessor

        ),

        (

            "model",

            RandomForestClassifier(

                n_estimators=100,

                max_depth=10,

                random_state=42,

                n_jobs=-1

            )

        )

    ]

)


random_forest.fit(
    X_train,
    y_train
)


rf_predictions = random_forest.predict(
    X_test
)


rf_probabilities = (

    random_forest
    .predict_proba(X_test)[:, 1]

)


print(
    "Random Forest completed."
)


# ============================================================
# 18. EVALUATION
# ============================================================

def calculate_results(

    model_name,

    actual,

    predictions,

    probabilities

):

    return {

        "Model": model_name,

        "Accuracy": accuracy_score(
            actual,
            predictions
        ),

        "Precision": precision_score(
            actual,
            predictions,
            zero_division=0
        ),

        "Recall": recall_score(
            actual,
            predictions,
            zero_division=0
        ),

        "F1 Score": f1_score(
            actual,
            predictions,
            zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            actual,
            probabilities
        )

    }


dt_results = calculate_results(

    "Decision Tree",

    y_test,

    dt_predictions,

    dt_probabilities

)


rf_results = calculate_results(

    "Random Forest",

    y_test,

    rf_predictions,

    rf_probabilities

)


# ============================================================
# 19. PRINT RESULTS
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "DECISION TREE RESULTS"
)

print(
    "=" * 70
)


for key, value in dt_results.items():

    if key != "Model":

        print(
            f"{key}: {value:.4f}"
        )


print(
    "\nClassification Report:"
)


print(

    classification_report(

        y_test,

        dt_predictions,

        target_names=[
            "No Churn",
            "Churn"
        ],

        zero_division=0

    )

)


print(
    "\n" + "=" * 70
)

print(
    "RANDOM FOREST RESULTS"
)

print(
    "=" * 70
)


for key, value in rf_results.items():

    if key != "Model":

        print(
            f"{key}: {value:.4f}"
        )


print(
    "\nClassification Report:"
)


print(

    classification_report(

        y_test,

        rf_predictions,

        target_names=[
            "No Churn",
            "Churn"
        ],

        zero_division=0

    )

)


# ============================================================
# 20. CONFUSION MATRIX FUNCTION
# ============================================================

def create_confusion_matrix(

    actual,

    predictions,

    title,

    filename

):

    matrix = confusion_matrix(

        actual,

        predictions

    )


    plt.figure(
        figsize=(6, 5)
    )


    plt.imshow(
        matrix
    )


    plt.title(
        title
    )


    plt.xlabel(
        "Predicted"
    )


    plt.ylabel(
        "Actual"
    )


    plt.xticks(
        [0, 1],
        ["No Churn", "Churn"]
    )


    plt.yticks(
        [0, 1],
        ["No Churn", "Churn"]
    )


    for i in range(2):

        for j in range(2):

            plt.text(

                j,

                i,

                matrix[i, j],

                ha="center",

                va="center"

            )


    plt.colorbar()


    plt.tight_layout()


    plt.savefig(

        os.path.join(

            VISUALIZATION_FOLDER,

            filename

        ),

        dpi=300

    )


    plt.close()


# ============================================================
# 21. CREATE CONFUSION MATRICES
# ============================================================

create_confusion_matrix(

    y_test,

    dt_predictions,

    "Decision Tree - Confusion Matrix",

    "confusion_matrix_decision_tree.png"

)


create_confusion_matrix(

    y_test,

    rf_predictions,

    "Random Forest - Confusion Matrix",

    "confusion_matrix_random_forest.png"

)


# ============================================================
# 22. ROC CURVE
# ============================================================

dt_fpr, dt_tpr, _ = roc_curve(

    y_test,

    dt_probabilities

)


rf_fpr, rf_tpr, _ = roc_curve(

    y_test,

    rf_probabilities

)


plt.figure(
    figsize=(8, 6)
)


plt.plot(

    dt_fpr,

    dt_tpr,

    label=(

        "Decision Tree "
        f"(AUC = {dt_results['ROC-AUC']:.3f})"

    )

)


plt.plot(

    rf_fpr,

    rf_tpr,

    label=(

        "Random Forest "
        f"(AUC = {rf_results['ROC-AUC']:.3f})"

    )

)


plt.plot(

    [0, 1],

    [0, 1],

    linestyle="--",

    label="Random Classifier"

)


plt.xlabel(
    "False Positive Rate"
)


plt.ylabel(
    "True Positive Rate"
)


plt.title(
    "ROC Curve - Customer Churn Prediction"
)


plt.legend()


plt.grid(
    True,
    alpha=0.3
)


plt.tight_layout()


plt.savefig(

    os.path.join(

        VISUALIZATION_FOLDER,

        "roc_curve.png"

    ),

    dpi=300

)


plt.close()


# ============================================================
# 23. MODEL COMPARISON
# ============================================================

results = pd.DataFrame(

    [

        dt_results,

        rf_results

    ]

)


print(
    "\n" + "=" * 70
)

print(
    "MODEL COMPARISON"
)

print(
    "=" * 70
)


print(

    results.round(4).to_string(
        index=False
    )

)


# ============================================================
# 24. MODEL COMPARISON GRAPH
# ============================================================

metrics = [

    "Accuracy",

    "Precision",

    "Recall",

    "F1 Score",

    "ROC-AUC"

]


x = range(
    len(metrics)
)


plt.figure(
    figsize=(10, 6)
)


plt.bar(

    [i - 0.2 for i in x],

    results.iloc[0][metrics],

    width=0.4,

    label="Decision Tree"

)


plt.bar(

    [i + 0.2 for i in x],

    results.iloc[1][metrics],

    width=0.4,

    label="Random Forest"

)


plt.xticks(
    list(x),
    metrics
)


plt.ylabel(
    "Score"
)


plt.ylim(
    0,
    1
)


plt.title(
    "Machine Learning Model Performance Comparison"
)


plt.legend()


plt.grid(
    axis="y",
    alpha=0.3
)


plt.tight_layout()


plt.savefig(

    os.path.join(

        VISUALIZATION_FOLDER,

        "model_comparison.png"

    ),

    dpi=300

)


plt.close()


# ============================================================
# 25. FEATURE IMPORTANCE
# ============================================================

rf_preprocessor = (

    random_forest
    .named_steps["preprocessor"]

)


rf_model = (

    random_forest
    .named_steps["model"]

)


feature_names = []


if categorical_features:

    categorical_encoder = (

        rf_preprocessor
        .named_transformers_["categorical"]

    )


    categorical_names = (

        categorical_encoder
        .get_feature_names_out(
            categorical_features
        )

    )


    feature_names.extend(
        categorical_names
    )


feature_names.extend(
    numerical_features
)


importances = (
    rf_model.feature_importances_
)


feature_importance_df = pd.DataFrame(

    {

        "Feature": feature_names,

        "Importance": importances

    }

)


feature_importance_df = (

    feature_importance_df
    .sort_values(
        by="Importance",
        ascending=False
    )

)


print(
    "\n" + "=" * 70
)

print(
    "TOP 15 IMPORTANT FEATURES"
)

print(
    "=" * 70
)


print(

    feature_importance_df
    .head(15)
    .to_string(
        index=False
    )

)


# ============================================================
# 26. FEATURE IMPORTANCE GRAPH
# ============================================================

top_features = (

    feature_importance_df
    .head(15)
    .sort_values(
        by="Importance"
    )

)


plt.figure(
    figsize=(10, 7)
)


plt.barh(

    top_features["Feature"],

    top_features["Importance"]

)


plt.xlabel(
    "Importance"
)


plt.ylabel(
    "Feature"
)


plt.title(
    "Top 15 Features - Random Forest"
)


plt.tight_layout()


plt.savefig(

    os.path.join(

        VISUALIZATION_FOLDER,

        "feature_importance.png"

    ),

    dpi=300

)


plt.close()


# ============================================================
# 27. SAVE RESULTS
# ============================================================

results.to_csv(

    os.path.join(

        VISUALIZATION_FOLDER,

        "model_results.csv"

    ),

    index=False

)


# ============================================================
# 28. FINAL RESULT
# ============================================================

if (

    rf_results["ROC-AUC"]

    >=

    dt_results["ROC-AUC"]

):

    best_model = "Random Forest"

    best_score = rf_results["ROC-AUC"]

else:

    best_model = "Decision Tree"

    best_score = dt_results["ROC-AUC"]


print(
    "\n" + "=" * 70
)

print(
    "WEEK 2 PREDICTIVE MODELING COMPLETED"
)

print(
    "=" * 70
)


print(
    "\nBest Model:",
    best_model
)


print(
    f"Best ROC-AUC Score: {best_score:.4f}"
)


print(
    "\nFiles generated inside visualization:"
)


print(
    "1. confusion_matrix_decision_tree.png"
)

print(
    "2. confusion_matrix_random_forest.png"
)

print(
    "3. roc_curve.png"
)

print(
    "4. model_comparison.png"
)

print(
    "5. feature_importance.png"
)

print(
    "6. model_results.csv"
)


print(
    "\nProject completed successfully."
)

TELCO CUSTOMER CHURN
WEEK 2 - PREDICTIVE MODELING

Searching for the Telco Customer Churn dataset...

Dataset selected:
c:\Users\B.P VEDHAVARSHINE\Documents\OneDrive\Documents\telco-customer-churn-analysis prediction\data\Telco_Customer_Churn_Cleaned.csv

Dataset loaded successfully.
Rows: 7043
Columns: 33

Dataset columns:
['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']

Churn column detected:
Churn Label

Original churn values:
Churn Label
No     5174
Yes    1869
Name: count, dtype: int64

Final churn distribution:
Churn_Target
0    5174
1    1869
N